# 워크플로 소개 — Workflows vs Agents

**Skilljar Lesson 01 대응**

이 노트북에서 다루는 내용:
1. 워크플로(Workflow)와 에이전트(Agent)의 개념적 차이
2. Evaluator-Optimizer 패턴 소개
3. 단순성 우선 원칙 (Simplicity First)

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## §1. 워크플로 vs 에이전트 개념

- **워크플로 (Workflow)**: LLM 호출이 **사전 정의된 코드 경로**를 따라 실행. 개발자가 순서와 분기를 결정.
- **에이전트 (Agent)**: LLM이 **스스로 다음 행동을 결정**. 어떤 도구를, 몇 번, 어떤 순서로 호출할지 LLM이 판단.

| 구분 | 워크플로 | 에이전트 |
|------|----------|----------|
| 제어 주체 | 코드 (개발자) | LLM (모델) |
| 예측 가능성 | 높음 | 낮음 |
| 유연성 | 낮음 | 높음 |
| 디버깅 | 쉬움 | 어려움 |

## §2. 가장 단순한 형태: 단일 LLM 호출

워크플로나 에이전트 이전에, **단일 호출**로 충분한지 먼저 확인합니다.

In [ ]:
# 단일 LLM 호출 — 가장 단순한 형태
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{
        "role": "user",
        "content": "Python의 리스트 컴프리헨션을 간단히 설명해줘."
    }]
)
print(response.content[0].text)

## §3. Evaluator-Optimizer 패턴

워크플로의 가장 기본 패턴 중 하나: **생성자(Generator)** 가 결과를 만들고, **평가자(Evaluator)** 가 품질을 검증합니다.

```
생성자 → 평가자 → (통과) → 완료
               → (불통과) → 생성자 (재시도)
```

In [ ]:
import json

def generate(prompt):
    """생성자: 주어진 프롬프트로 결과를 생성한다."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system="You are a helpful assistant. Write clear, concise responses.",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


def evaluate(original_prompt, generated_text):
    """평가자: 생성 결과의 품질을 1-10으로 평가한다."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        system=(
            "You are a quality evaluator. Rate the response quality "
            "on a scale of 1-10. Respond ONLY with JSON: "
            '{"score": N, "feedback": "brief reason"}'
        ),
        messages=[{
            "role": "user",
            "content": f"Original prompt: {original_prompt}\n\n"
                       f"Response to evaluate:\n{generated_text}"
        }]
    )
    return json.loads(response.content[0].text)

In [ ]:
def evaluator_optimizer(prompt, threshold=7, max_retries=3):
    """Evaluator-Optimizer 패턴: 품질 기준을 통과할 때까지 반복한다."""
    for attempt in range(1, max_retries + 1):
        print(f"\n--- Attempt {attempt} ---")
        
        # 생성
        result = generate(prompt)
        print(f"생성 결과: {result[:100]}...")
        
        # 평가
        evaluation = evaluate(prompt, result)
        print(f"평가: score={evaluation['score']}, feedback={evaluation['feedback']}")
        
        if evaluation["score"] >= threshold:
            print(f"\n✅ 품질 기준 통과 (score={evaluation['score']} >= {threshold})")
            return result
        else:
            print(f"❌ 기준 미달 (score={evaluation['score']} < {threshold}), 재시도...")
            # 피드백을 포함하여 재생성
            prompt = (
                f"{prompt}\n\n"
                f"Previous attempt feedback: {evaluation['feedback']}\n"
                f"Please improve based on this feedback."
            )
    
    print(f"\n⚠️ 최대 재시도 횟수 도달 ({max_retries}회)")
    return result

In [ ]:
# 실행
final = evaluator_optimizer(
    "건축공학에서 RC 보의 설계 순서를 3단계로 설명해줘.",
    threshold=7,
    max_retries=3
)
print("\n=== 최종 결과 ===")
print(final)

## §4. 핵심 정리

- **단순성 우선**: 단일 호출 → 워크플로 → 에이전트 순으로 복잡성을 높여간다
- **Evaluator-Optimizer**: 생성 → 평가 → (필요시 재생성)의 워크플로 패턴
- **다음 노트북**: `S8_02_parallelization.ipynb`에서 병렬화 워크플로를 구현한다